In [1]:
!pip install langchain-chroma chromadb langchain-core langchain-community langchain pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/2

In [2]:
from google.colab import files

uploaded = files.upload()  # file picker opens → select all 4 PDFs

Saving PDF3_Hardware_Equipment_Guide 1.pdf to PDF3_Hardware_Equipment_Guide 1.pdf
Saving PDF2_Service_Outage_Guide 1.pdf to PDF2_Service_Outage_Guide 1.pdf
Saving PDF1_Network_Connectivity_Guide 1.pdf to PDF1_Network_Connectivity_Guide 1.pdf
Saving PDF4_Customer_Experience_Guide 1.pdf to PDF4_Customer_Experience_Guide 1.pdf


In [3]:
import os
for f in os.listdir("/content"):
    if f.endswith(".pdf"):
        print(f)

PDF2_Service_Outage_Guide 1.pdf
PDF3_Hardware_Equipment_Guide 1.pdf
PDF1_Network_Connectivity_Guide 1.pdf
PDF4_Customer_Experience_Guide 1.pdf


In [4]:
from langchain_community.document_loaders import PyPDFLoader

pdf_files = {
    "service_outage":   "/content/PDF2_Service_Outage_Guide 1.pdf",
    "hardware":         "/content/PDF3_Hardware_Equipment_Guide 1.pdf",
    "network":          "/content/PDF1_Network_Connectivity_Guide 1.pdf",
    "customer_experience": "/content/PDF4_Customer_Experience_Guide 1.pdf"
}

all_docs = []

for category, pdf_path in pdf_files.items():
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    for doc in docs:
        doc.metadata["category"] = category
    all_docs.extend(docs)
    print(f"✅ {category} → {len(docs)} pages loaded")

print(f"\nTotal pages: {len(all_docs)}")

✅ service_outage → 3 pages loaded
✅ hardware → 3 pages loaded
✅ network → 3 pages loaded
✅ customer_experience → 3 pages loaded

Total pages: 12


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

all_splits = text_splitter.split_documents(all_docs)
print(f"Total chunks created: {len(all_splits)}")

# Check a sample chunk
print("\n--- Sample Chunk ---")
print("Content  :", all_splits[0].page_content[:300])
print("Metadata :", all_splits[0].metadata)

Total chunks created: 29

--- Sample Chunk ---
Content  : Service Outage
 What to do when services stop working — explained simply for customers
1. Complete Voice & Data Down
What this means: You cannot make or receive any calls and mobile internet has completely stopped.
Step 1: Do not panic — this is usually a temporary outage in your area affecting many
Metadata : {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-22T15:03:51+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-22T15:03:51+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/content/PDF2_Service_Outage_Guide 1.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'category': 'service_outage'}


In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded!")

/tmp/ipykernel_1951/1798636551.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!


In [7]:
from langchain_chroma import Chroma

CHROMA_PATH = "/content/telecom_chroma_db"

vectorstore = Chroma.from_documents(
    documents=all_splits,
    embedding=embeddings,
    persist_directory=CHROMA_PATH,
    collection_name="telecom_tickets"
)

print(f"✅ ChromaDB created!")
print(f"   Total vectors stored: {vectorstore._collection.count()}")

✅ ChromaDB created!
   Total vectors stored: 29


In [8]:
# Test 1: Search across ALL categories
print("=== Search across all categories ===")
results = vectorstore.similarity_search("internet not working", k=3)
for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print(f"  Category : {doc.metadata['category']}")
    print(f"  Content  : {doc.page_content[:200]}")

# Test 2: Search within a SPECIFIC category
print("\n=== Search within 'network' category only ===")
results = vectorstore.similarity_search(
    "connection dropping",
    k=3,
    filter={"category": "network"}
)
for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print(f"  Category : {doc.metadata['category']}")
    print(f"  Content  : {doc.page_content[:200]}")

=== Search across all categories ===

Result 1
  Category : network
  Content  : What this means: Your home broadband or fiber internet has completely stopped working, often after
bad weather or construction nearby.
Step 1: Check if the lights on your router/modem are normal. If t

Result 2
  Category : hardware
  Content  : ■■ Power failures are usually resolved within 4 hours. Our sites have backup batteries for short
outages.
3. Router / Modem Not Working
What this means: Your home broadband router or modem is showing 

Result 3
  Category : hardware
  Content  : Step 4: Try connecting your phone directly to the router via Wi-Fi to test if it works.
Step 5: If lights are normal but internet still does not work, restart your phone or laptop too.
Step 6: If the 

=== Search within 'network' category only ===

Result 1
  Category : network
  Content  : Network & Connectivity
 Simple troubleshooting guide for customers — no technical knowledge needed
1. Signal Loss / Weak Coverage
What 

In [9]:
import shutil
from google.colab import files

# Zip the folder
shutil.make_archive("/content/telecom_chroma_db", "zip", "/content/telecom_chroma_db")
print("✅ Zipped!")

# Download to your local machine
files.download("/content/telecom_chroma_db.zip")
print("✅ Download started!")

✅ Zipped!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!
